In [ ]:
!pip install lightgbm tensorflow scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
print("Path:", path)

In [ ]:
import os

os.listdir(path)

In [ ]:
os.listdir(f"{path}/m5-forecasting-accuracy")

In [ ]:
import pandas as pd

In [ ]:
sales = pd.read_csv(f"{path}/m5-forecasting-accuracy/sales_train_validation.csv")
calendar = pd.read_csv(f"{path}/m5-forecasting-accuracy/calendar.csv")
prices = pd.read_csv(f"{path}/m5-forecasting-accuracy/sell_prices.csv")

In [68]:
import os

dataset_path = os.path.join(path, "m5-forecasting-accuracy")

sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
print(sales.shape)
print(calendar.shape)
print(prices.shape)

In [ ]:
print(path)
os.listdir(path)

In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
sales = sales[
    (sales['state_id'].isin(['CA','TX','WI'])) &
    (sales['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

sales = sales.sample(frac=0.5, random_state=42)


day_cols = [c for c in sales.columns if 'd_' in c]
half_days = day_cols[:len(day_cols)//2]

sales = sales[['item_id','dept_id','cat_id','store_id','state_id'] + half_days]

In [ ]:
sales_long = sales.melt(
    id_vars=['item_id','dept_id','cat_id','store_id','state_id'],
    var_name='d',
    value_name='sales'
)

In [ ]:
sales_long = sales_long.merge(
    calendar[['d', 'wm_yr_wk', 'month']],
    on='d',
    how='left'
)

In [ ]:
sales_long = sales_long.merge(
    prices,
    on=['store_id','item_id','wm_yr_wk'],
    how='left'
)

In [ ]:
sales_long['revenue'] = sales_long['sales'] * sales_long['sell_price']

In [ ]:
weekly_df = sales_long.groupby(
    ['cat_id', 'state_id', 'wm_yr_wk', 'month'],
    as_index=False
)['revenue'].sum()

In [ ]:
del sales_long
import gc
gc.collect()

In [ ]:
df = weekly_df.copy()

In [ ]:
df = df.sort_values(
    ['cat_id', 'state_id', 'wm_yr_wk']
)

for lag in [1, 2, 3, 4, 8, 12]:
    df[f'lag_{lag}'] = (
        df.groupby(['cat_id', 'state_id'])['revenue']
          .shift(lag)
    )

df['rolling_mean_4'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(4).mean())
)

df['rolling_mean_12'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(12).mean())
)

df = df.dropna()

In [ ]:
features = [
    'lag_1',
    'lag_2',
    'lag_3',
    'lag_4',
    'lag_8',
    'lag_12',
    'rolling_mean_4',
    'rolling_mean_12',
    'month'
]

X = df[features]
y = df['revenue']

In [ ]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_val   = X.iloc[split:]

y_train = y.iloc[:split]
y_val   = y.iloc[split:]

In [ ]:
model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

model_lgb.fit(
    X_train,
    y_train
)

pred_lgb = model_lgb.predict(X_val)

In [ ]:
lookback = 10

values = df["revenue"].values

X_lstm = []
y_lstm = []

for i in range(lookback, len(values)):
    X_lstm.append(values[i-lookback:i])
    y_lstm.append(values[i])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

X_lstm = X_lstm.reshape(
    X_lstm.shape[0],
    X_lstm.shape[1],
    1
)

In [ ]:
split_lstm = int(len(X_lstm) * 0.8)

X_lstm_train = X_lstm[:split_lstm]
X_lstm_val   = X_lstm[split_lstm:]

y_lstm_train = y_lstm[:split_lstm]
y_lstm_val   = y_lstm[split_lstm:]

In [ ]:
model_lstm = Sequential([
    LSTM(64, input_shape=(lookback, 1)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model_lstm.compile(
    optimizer="adam",
    loss="mse"
)

history = model_lstm.fit(
    X_lstm_train,
    y_lstm_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
pred_lstm = model_lstm.predict(
    X_lstm_val
).flatten()

In [ ]:
actual = y_val_aligned

pred_lgb = pred_lgb_aligned
pred_lstm = pred_lstm_aligned


In [ ]:
best_weight = None
best_rmse = float("inf")

for w in np.arange(0, 1.01, 0.01):

    hybrid_pred = (
        w * pred_lgb
        + (1 - w) * pred_lstm
    )

    rmse = np.sqrt(
        mean_squared_error(actual, hybrid_pred)
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_weight = w

print("Best LightGBM weight:", best_weight)
print("Best LSTM weight:", 1 - best_weight)
print("Best RMSE:", best_rmse)

In [ ]:
hybrid_pred = (
    best_weight * pred_lgb
    + (1 - best_weight) * pred_lstm
)

In [ ]:
mae = mean_absolute_error(actual, hybrid_pred)

rmse = np.sqrt(
    mean_squared_error(actual, hybrid_pred)
)

mape = np.mean(
    np.abs((actual - hybrid_pred) / actual)
) * 100

print("Hybrid MAE :", mae)
print("Hybrid RMSE:", rmse)
print("Hybrid MAPE:", mape)